# Bootstrap Confidence Intervals for mAP50 Comparison

This notebook implements paired bootstrap resampling to compute confidence intervals for:
1. Individual model mAP50 scores
2. The difference in mAP50 between baseline and fine-tuned models

## Methodology

We treat mAP50 as a statistic computed on a finite test set. The bootstrap procedure:
- Resamples documents/images with replacement (same sample for both models)
- Recomputes mAP50 for each bootstrap iteration
- Uses percentile method to construct confidence intervals
- Naturally preserves pairing and correlation between models

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from ultralytics.utils.metrics import ConfusionMatrix, ap_per_class, box_iou

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load Model Predictions

Load predictions from both models on the test set. We need the per-image predictions to enable resampling.

In [23]:
def load_predictions(model_path, data_yaml, split='test'):
    """
    Load model and get predictions on test set.
    Returns predictions and ground truth for each image.
    """
    model = YOLO(str(model_path))
    
    # Run validation to get predictions
    results = model.val(
        data=data_yaml,
        split=split,
        batch=16,
        imgsz=640,
        verbose=False,
        save_json=True  # Save predictions for bootstrap
    )
    
    return results

def extract_per_image_data(results):
    """
    Extract per-image predictions and ground truth.
    Returns structured data for bootstrap resampling.
    """
    # Access the underlying validator which has per-image data
    stats = results.speed  # Access timing
    
    # Get per-image statistics from results
    # YOLO stores: predictions (predn), labels (labelsn), and other per-image data
    per_image_data = {
        'predictions': [],  # List of prediction arrays per image
        'labels': [],       # List of label arrays per image
        'image_ids': []     # Image identifiers
    }
    
    return per_image_data, results

In [24]:
# Configuration
data_yaml = '../src/training/finance-image-parser.yaml'
baseline_model_path = Path('../models/pretrained/yolov8n.pt')
finetuned_model_path = Path('../models/experiments/final/yolo-final-20251123/weights/best.pt')

print("Loading baseline model predictions...")
baseline_results = load_predictions(baseline_model_path, data_yaml)
baseline_map50 = float(baseline_results.box.map50)

print("\nLoading fine-tuned model predictions...")
finetuned_results = load_predictions(finetuned_model_path, data_yaml)
finetuned_map50 = float(finetuned_results.box.map50)

print(f"\nObserved mAP50:")
print(f"  Baseline:    {baseline_map50:.4f} ({baseline_map50*100:.2f}%)")
print(f"  Fine-tuned:  {finetuned_map50:.4f} ({finetuned_map50*100:.2f}%)")
print(f"  Improvement: {finetuned_map50 - baseline_map50:.4f} ({(finetuned_map50 - baseline_map50)*100:.2f}%)")

Ultralytics YOLOv8.0.196  Python-3.11.14 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)


Loading baseline model predictions...


YOLOv8n summary (fused): 168 layers, 3151904 parameters, 0 gradients, 8.7 GFLOPs
val: Scanning D:\docs\MADS\699\data\input\testing\labels.cache... 481 images, 0 backgrounds, 0 corrupt: 100%|██████████| 481/481 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 31/31 [00:05<00:00,  5.98it/s]
                   all        481       1237    0.00272    0.00984    0.00141   0.000467
Speed: 0.4ms preprocess, 3.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Saving runs\detect\val13\predictions.json...
Results saved to runs\detect\val13
c:\Users\Leo\miniconda3\envs\capstone\Lib\site-packages\ultralytics\nn\tasks.py:567: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/S


Loading fine-tuned model predictions...


Model summary (fused): 168 layers, 3006233 parameters, 0 gradients, 8.1 GFLOPs
val: Scanning D:\docs\MADS\699\data\input\testing\labels.cache... 481 images, 0 backgrounds, 0 corrupt: 100%|██████████| 481/481 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 31/31 [00:37<00:00,  1.20s/it]
                   all        481       1237      0.889       0.78       0.87      0.691
Speed: 0.6ms preprocess, 3.8ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving runs\detect\val14\predictions.json...
Results saved to runs\detect\val14



Observed mAP50:
  Baseline:    0.0014 (0.14%)
  Fine-tuned:  0.8699 (86.99%)
  Improvement: 0.8685 (86.85%)


## 2. Alternative Approach: Dataset-Level Bootstrap

Since extracting per-image predictions from YOLO's internal structure is complex, we'll use a dataset-level bootstrap approach:
- Create bootstrap samples by resampling image indices
- Create temporary dataset configurations for each bootstrap sample
- Run validation on each bootstrap sample

This is more computationally intensive but ensures correct mAP50 calculation.

In [25]:
def get_test_images(data_yaml):
    """
    Get list of test images from dataset configuration.
    """
    import yaml
    
    with open(data_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Get the root path and test path from YAML
    root_path = Path(data_config.get('path', ''))
    test_rel_path = data_config.get('test', data_config.get('val'))
    
    # Construct full path: root + test path
    if root_path.is_absolute():
        test_path = root_path / test_rel_path
    else:
        yaml_dir = Path(data_yaml).parent.resolve()
        test_path = yaml_dir / root_path / test_rel_path
    
    test_path = test_path.resolve()
    
    print(f"Dataset root: {root_path}")
    print(f"Test path: {test_rel_path}")
    print(f"Full test images path: {test_path}")
    
    # Check if path exists
    if not test_path.exists():
        raise FileNotFoundError(
            f"Test images directory not found at: {test_path}\n"
            f"Root path: {root_path}\n"
            f"Test relative path: {test_rel_path}"
        )
    
    # Check if it's a text file listing images
    if test_path.is_file() and test_path.suffix == '.txt':
        with open(test_path, 'r') as f:
            image_paths = [line.strip() for line in f.readlines() if line.strip()]
        # Make paths absolute if needed
        abs_paths = []
        for img_path in image_paths:
            p = Path(img_path)
            if not p.is_absolute():
                p = root_path / p
            abs_paths.append(str(p.resolve()))
        return abs_paths
    
    # It's a directory - get all image files
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    images = []
    for ext in image_extensions:
        images.extend(test_path.glob(f'*{ext}'))
        images.extend(test_path.glob(f'*{ext.upper()}'))
    
    if not images:
        raise FileNotFoundError(f"No images found in {test_path}")
    
    print(f"Found {len(images)} test images")
    return sorted([str(img) for img in images])


# Load test image paths
test_images = get_test_images(data_yaml)
n_images = len(test_images)
print(f"Found {n_images} test images")

Dataset root: D:\docs\MADS\699\data\input
Test path: testing/images
Full test images path: D:\docs\MADS\699\data\input\testing\images
Found 962 test images
Found 962 test images


## 3. Simplified Bootstrap: Prediction-Based Resampling

A more efficient approach: Load predictions once, then bootstrap at the prediction level.
We'll use the predictions stored by YOLO and compute mAP50 by resampling them.

In [26]:
def load_model_predictions_detailed(model_path, data_yaml, split='test'):
    """
    Load detailed predictions for bootstrap analysis.
    Returns per-image predictions and labels.
    """
    from ultralytics.utils import LOGGER
    import torch
    
    model = YOLO(str(model_path))
    
    # Get validator
    validator = model.val(
        data=data_yaml,
        split=split,
        batch=16,
        imgsz=640,
        verbose=False
    )
    
    # Extract stored statistics
    # YOLO validation stores stats as list of tuples per batch
    # Each entry: (correct_predictions, pred_confidence, pred_class, target_class)
    
    return validator


def compute_map50_from_stats(stats, nc=3):
    """
    Compute mAP50 from detection statistics.
    
    Args:
        stats: List of (correct, conf, pcls, tcls) tuples per image
        nc: Number of classes
    """
    import torch
    from ultralytics.utils.metrics import ap_per_class
    
    if not stats:
        return 0.0
    
    # Concatenate stats across all images
    stats = [torch.cat(x, 0).cpu().numpy() for x in zip(*stats)]
    
    if len(stats) and stats[0].any():
        # Compute AP per class
        tp, fp, p, r, f1, ap, ap_class = ap_per_class(
            *stats,
            plot=False,
            save_dir='',
            names={}
        )
        
        # mAP50 is ap[:, 0] (AP at IoU=0.5)
        map50 = ap[:, 0].mean() if len(ap) else 0.0
        return float(map50)
    
    return 0.0

## 4. Practical Bootstrap Implementation

Since accessing YOLO's internal validation statistics is complex, we'll use a practical approach:
- Store predictions and labels in a structured format
- Implement our own mAP50 calculation
- Bootstrap by resampling images and recalculating metrics

In [27]:
def get_predictions_per_image(model, data_yaml, split='test'):
    """
    Run inference and collect per-image predictions.
    """
    import yaml
    from ultralytics.data.dataset import YOLODataset
    import torch
    
    # Load data config
    with open(data_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Get test path
    root_path = Path(data_config.get('path', ''))
    test_rel_path = data_config.get(split, data_config.get('val'))
    
    if root_path.is_absolute():
        test_path = root_path / test_rel_path
    else:
        yaml_dir = Path(data_yaml).parent.resolve()
        test_path = yaml_dir / root_path / test_rel_path
    
    # Run predictions
    results_list = model.predict(
        source=str(test_path),
        imgsz=640,
        conf=0.001,  # Low threshold to capture all detections
        iou=0.6,
        verbose=False
    )
    
    predictions = []
    for r in results_list:
        # Extract boxes, scores, classes
        boxes = r.boxes.xyxy.cpu().numpy() if len(r.boxes) > 0 else np.array([]).reshape(0, 4)
        scores = r.boxes.conf.cpu().numpy() if len(r.boxes) > 0 else np.array([])
        classes = r.boxes.cls.cpu().numpy() if len(r.boxes) > 0 else np.array([])
        
        predictions.append({
            'boxes': boxes,
            'scores': scores,
            'classes': classes,
            'path': r.path
        })
    
    return predictions


def load_ground_truth(data_yaml, split='test'):
    """
    Load ground truth labels for test set.
    """
    import yaml
    with open(data_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Get the root path and test path from YAML
    root_path = Path(data_config.get('path', ''))
    test_rel_path = data_config.get(split, data_config.get('val'))
    
    # Construct full path to images directory
    if root_path.is_absolute():
        images_path = root_path / test_rel_path
    else:
        yaml_dir = Path(data_yaml).parent.resolve()
        images_path = yaml_dir / root_path / test_rel_path
    
    images_path = images_path.resolve()
    
    # Replace 'images' with 'labels' in the path
    labels_path = Path(str(images_path).replace('images', 'labels'))
    
    print(f"Images path: {images_path}")
    print(f"Labels path: {labels_path}")
    
    # Check if labels path exists
    if not labels_path.exists():
        # Try alternative: go up from images and look for labels
        if 'images' in images_path.parts:
            # Find the 'images' part and replace with 'labels'
            parts = list(images_path.parts)
            for i, part in enumerate(parts):
                if part == 'images':
                    parts[i] = 'labels'
                    break
            labels_path = Path(*parts)
            print(f"Trying alternative labels path: {labels_path}")
    
    if not labels_path.exists():
        raise FileNotFoundError(
            f"Labels directory not found!\n"
            f"Tried: {labels_path}\n"
            f"Please verify your dataset structure."
        )
    
    # Load all label files
    ground_truth = {}
    label_files = list(labels_path.glob('*.txt'))
    
    print(f"Found {len(label_files)} label files")
    
    if len(label_files) == 0:
        raise FileNotFoundError(
            f"No .txt label files found in {labels_path}\n"
            f"Please check your dataset structure."
        )
    
    for label_file in label_files:
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        boxes = []
        classes = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(parts[0])
                # YOLO format: class x_center y_center width height (normalized)
                x, y, w, h = map(float, parts[1:5])
                boxes.append([x, y, w, h])
                classes.append(cls)
        
        ground_truth[label_file.stem] = {
            'boxes': np.array(boxes) if boxes else np.array([]).reshape(0, 4),
            'classes': np.array(classes) if classes else np.array([])
        }
    
    return ground_truth


print("Loading ground truth labels...")
ground_truth = load_ground_truth(data_yaml, split='test')
print(f"✓ Loaded ground truth for {len(ground_truth)} images")
print(f"Total ground truth boxes: {sum(len(gt['boxes']) for gt in ground_truth.values())}")

Loading ground truth labels...
Images path: D:\docs\MADS\699\data\input\testing\images
Labels path: D:\docs\MADS\699\data\input\testing\labels
Found 481 label files
✓ Loaded ground truth for 481 images
Total ground truth boxes: 1237


In [ ]:
print("Loading baseline predictions...")
baseline_model = YOLO(str(baseline_model_path))
baseline_preds = get_predictions_per_image(baseline_model, data_yaml, split='test')
print(f"✓ Loaded {len(baseline_preds)} baseline predictions")

print("\nLoading fine-tuned predictions...")
finetuned_model = YOLO(str(finetuned_model_path))
finetuned_preds = get_predictions_per_image(finetuned_model, data_yaml, split='test')
print(f"✓ Loaded {len(finetuned_preds)} fine-tuned predictions")

# Verify predictions and ground truth alignment
print(f"\nVerification:")
print(f"  Number of images with predictions: {len(baseline_preds)}")
print(f"  Number of images with ground truth: {len(ground_truth)}")
print(f"  Baseline total detections: {sum(len(p['boxes']) for p in baseline_preds)}")
print(f"  Fine-tuned total detections: {sum(len(p['boxes']) for p in finetuned_preds)}")

# Check if filenames match
sample_pred_stems = set([Path(p['path']).stem for p in baseline_preds[:10]])
sample_gt_stems = set(list(ground_truth.keys())[:10])
print(f"\nSample prediction stems: {list(sample_pred_stems)[:3]}")
print(f"Sample ground truth stems: {list(sample_gt_stems)[:3]}")

if not sample_pred_stems.intersection(sample_gt_stems):
    print("\n⚠ WARNING: Prediction and ground truth filenames don't seem to match!")
    print("This may cause issues with the bootstrap calculation.")

c:\Users\Leo\miniconda3\envs\capstone\Lib\site-packages\ultralytics\nn\tasks.py:567: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(file, map_location='cpu'

Loading baseline predictions...
✓ Loaded 481 baseline predictions

Loading fine-tuned predictions...


## 5. Bootstrap Confidence Intervals
Now we implement the paired bootstrap procedure.

In [ ]:
def compute_iou_xywh(box1, box2):
    """
    Compute IoU between two boxes in YOLO format (x_center, y_center, width, height).
    All values are normalized [0, 1].
    """
    # Convert from center format to corner format
    box1_x1 = box1[0] - box1[2] / 2
    box1_y1 = box1[1] - box1[3] / 2
    box1_x2 = box1[0] + box1[2] / 2
    box1_y2 = box1[1] + box1[3] / 2

    box2_x1 = box2[0] - box2[2] / 2
    box2_y1 = box2[1] - box2[3] / 2
    box2_x2 = box2[0] + box2[2] / 2
    box2_y2 = box2[1] + box2[3] / 2

    # Compute intersection
    inter_x1 = max(box1_x1, box2_x1)
    inter_y1 = max(box1_y1, box2_y1)
    inter_x2 = min(box1_x2, box2_x2)
    inter_y2 = min(box1_y2, box2_y2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Compute union
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - inter_area

    # Compute IoU
    iou = inter_area / union_area if union_area > 0 else 0
    return iou


def compute_iou_xyxy(box1, box2):
    """
    Compute IoU between two boxes in corner format (x1, y1, x2, y2).
    """
    # Compute intersection
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Compute union
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area

    # Compute IoU
    iou = inter_area / union_area if union_area > 0 else 0
    return iou


def xywh_to_xyxy(box):
    """
    Convert YOLO format (x_center, y_center, width, height) to (x1, y1, x2, y2).
    Assumes normalized coordinates [0, 1].
    """
    x_center, y_center, width, height = box
    x1 = x_center - width / 2
    y1 = y_center - height / 2
    x2 = x_center + width / 2
    y2 = y_center + height / 2
    return np.array([x1, y1, x2, y2])


def xyxy_to_xywh(box):
    """
    Convert xyxy format to xywh (center format).
    Works with both normalized and pixel coordinates.
    """
    x1, y1, x2, y2 = box
    x_center = (x1 + x2) / 2
    y_center = (y1 + y2) / 2
    width = x2 - x1
    height = y2 - y1
    return np.array([x_center, y_center, width, height])


def compute_ap_per_class(tp, conf, pred_cls, target_cls, eps=1e-16):
    """
    Compute Average Precision per class.

    Args:
        tp: True positives array (n_predictions,)
        conf: Confidence scores (n_predictions,)
        pred_cls: Predicted classes (n_predictions,)
        target_cls: Target classes from ground truth
        eps: Small epsilon for numerical stability

    Returns:
        ap: Average precision per class
        p: Precision curve
        r: Recall curve
    """
    # Sort by confidence (descending)
    i = np.argsort(-conf)
    tp = tp[i]
    conf = conf[i]
    pred_cls = pred_cls[i]

    # Find unique classes
    unique_classes = np.unique(target_cls)
    n_classes = unique_classes.shape[0]

    # Initialize
    ap = np.zeros(n_classes)
    p_curve = []
    r_curve = []

    for ci, c in enumerate(unique_classes):
        # Select predictions for this class
        i_class = pred_cls == c
        n_gt = (target_cls == c).sum()  # Number of ground truth for this class
        n_p = i_class.sum()  # Number of predictions for this class

        if n_p == 0 or n_gt == 0:
            continue

        # Cumulative sum of TPs and FPs
        tp_class = tp[i_class]
        fp_class = 1 - tp_class

        tp_cumsum = np.cumsum(tp_class)
        fp_cumsum = np.cumsum(fp_class)

        # Compute precision and recall
        recall = tp_cumsum / (n_gt + eps)
        precision = tp_cumsum / (tp_cumsum + fp_cumsum + eps)

        # Compute AP using 101-point interpolation (COCO style)
        ap[ci] = compute_ap_from_pr(recall, precision)

        p_curve.append(precision)
        r_curve.append(recall)

    return ap, p_curve, r_curve


def compute_ap_from_pr(recall, precision):
    """
    Compute Average Precision from precision-recall curve.
    Uses 101-point interpolation (COCO style).
    """
    # Add sentinel values
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))

    # Compute the precision envelope
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))

    # Integrate area under curve using 101-point interpolation
    x = np.linspace(0, 1, 101)
    ap = np.trapz(np.interp(x, mrec, mpre), x)

    return ap


def compute_map50_resampled(predictions, ground_truth, indices):
    """
    Compute mAP50 on resampled predictions.

    Args:
        predictions: List of prediction dicts with 'boxes' (xyxy pixel), 'scores', 'classes', 'path'
        ground_truth: Dict mapping image stem to ground truth boxes (xywh normalized) / classes
        indices: Bootstrap sample indices

    Returns:
        mAP50 score
    """
    from PIL import Image
    
    iou_threshold = 0.5

    all_tp = []
    all_conf = []
    all_pred_cls = []
    all_target_cls = []

    # Process each image in the bootstrap sample
    for idx in indices:
        pred = predictions[idx]

        # Get image stem (filename without extension)
        img_path = Path(pred['path'])
        img_stem = img_path.stem

        # Get ground truth for this image
        if img_stem not in ground_truth:
            # No ground truth - all predictions are FP
            if len(pred['scores']) > 0:
                all_tp.extend([0] * len(pred['scores']))
                all_conf.extend(pred['scores'])
                all_pred_cls.extend(pred['classes'])
            continue

        gt = ground_truth[img_stem]
        gt_boxes = gt['boxes']  # YOLO format (x_center, y_center, w, h) normalized [0, 1]
        gt_classes = gt['classes']

        if len(gt_boxes) == 0:
            # No ground truth - all predictions are FP
            if len(pred['scores']) > 0:
                all_tp.extend([0] * len(pred['scores']))
                all_conf.extend(pred['scores'])
                all_pred_cls.extend(pred['classes'])
            continue

        # Get predictions (xyxy format, pixel coordinates from YOLO)
        pred_boxes = pred['boxes']
        pred_scores = pred['scores']
        pred_classes = pred['classes']

        if len(pred_boxes) == 0:
            # No predictions - add ground truth classes for recall calculation
            all_target_cls.extend(gt_classes)
            continue

        # Get image dimensions to normalize prediction boxes
        try:
            img = Image.open(img_path)
            img_width, img_height = img.size
        except:
            # If can't open image, assume standard size or skip
            # YOLO typically uses 640x640, but predictions should already be in pixel coords
            print(f"Warning: Could not open {img_path}, using default normalization")
            img_width, img_height = 640, 640

        # Track which ground truth boxes have been matched
        gt_matched = np.zeros(len(gt_boxes), dtype=bool)

        # Convert ground truth from xywh normalized to xyxy normalized for comparison
        gt_boxes_xyxy = np.array([xywh_to_xyxy(box) for box in gt_boxes])

        # Normalize prediction boxes (from pixel xyxy to normalized xyxy)
        pred_boxes_norm = pred_boxes.copy()
        pred_boxes_norm[:, [0, 2]] /= img_width  # Normalize x coordinates
        pred_boxes_norm[:, [1, 3]] /= img_height  # Normalize y coordinates

        # For each prediction, find best matching ground truth
        for pi, (pred_box_norm, pred_score, pred_cls) in enumerate(zip(pred_boxes_norm, pred_scores, pred_classes)):
            best_iou = 0
            best_gt_idx = -1

            for gi, (gt_box_xyxy, gt_cls) in enumerate(zip(gt_boxes_xyxy, gt_classes)):
                # Skip if already matched or wrong class
                if gt_matched[gi] or pred_cls != gt_cls:
                    continue

                # Compute IoU between normalized xyxy boxes
                iou = compute_iou_xyxy(pred_box_norm, gt_box_xyxy)

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gi

            # Check if prediction is TP or FP
            if best_iou >= iou_threshold:
                gt_matched[best_gt_idx] = True
                all_tp.append(1)
            else:
                all_tp.append(0)

            all_conf.append(pred_score)
            all_pred_cls.append(pred_cls)

        # Add ground truth classes for this image
        all_target_cls.extend(gt_classes)

    # Convert to numpy arrays
    if len(all_tp) == 0 or len(all_target_cls) == 0:
        return 0.0

    all_tp = np.array(all_tp)
    all_conf = np.array(all_conf)
    all_pred_cls = np.array(all_pred_cls)
    all_target_cls = np.array(all_target_cls)

    # Compute AP per class
    ap_per_cls, _, _ = compute_ap_per_class(all_tp, all_conf, all_pred_cls, all_target_cls)

    # mAP50 is the mean of AP across all classes
    map50 = ap_per_cls.mean() if len(ap_per_cls) > 0 else 0.0

    return float(map50)


def bootstrap_map50(
    baseline_predictions,
    finetuned_predictions,
    ground_truth,
    n_bootstrap=10000,
    confidence_level=0.95,
    random_seed=42
):
    """
    Compute bootstrap confidence intervals for mAP50 using paired resampling.
    
    Args:
        baseline_predictions: Predictions from baseline model
        finetuned_predictions: Predictions from fine-tuned model  
        ground_truth: Ground truth labels
        n_bootstrap: Number of bootstrap iterations
        confidence_level: Confidence level (default 0.95 for 95% CI)
        random_seed: Random seed for reproducibility
    
    Returns:
        Dictionary with bootstrap results and confidence intervals
    """
    np.random.seed(random_seed)
    
    n_images = len(baseline_predictions)
    image_indices = np.arange(n_images)
    
    # Storage for bootstrap statistics
    baseline_map50_bootstrap = np.zeros(n_bootstrap)
    finetuned_map50_bootstrap = np.zeros(n_bootstrap)
    delta_map50_bootstrap = np.zeros(n_bootstrap)
    
    print(f"Running {n_bootstrap} bootstrap iterations...")
    print("Note: This uses manual mAP50 calculation on resampled predictions.")
    print("For more accurate results, use bootstrap_with_yolo_validation() instead.\n")
    
    for b in tqdm(range(n_bootstrap)):
        # Sample images with replacement (same sample for both models)
        bootstrap_indices = np.random.choice(image_indices, size=n_images, replace=True)
        
        # Compute mAP50 on resampled data
        baseline_map50_bootstrap[b] = compute_map50_resampled(baseline_predictions, ground_truth, bootstrap_indices)
        finetuned_map50_bootstrap[b] = compute_map50_resampled(finetuned_predictions, ground_truth, bootstrap_indices)
        delta_map50_bootstrap[b] = finetuned_map50_bootstrap[b] - baseline_map50_bootstrap[b]
    
    # Compute confidence intervals using percentile method
    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2) * 100
    upper_percentile = (1 - alpha / 2) * 100
    
    results = {
        'baseline': {
            'point_estimate': baseline_map50,
            'bootstrap_mean': np.mean(baseline_map50_bootstrap),
            'bootstrap_distribution': baseline_map50_bootstrap,
            'ci_lower': np.percentile(baseline_map50_bootstrap, lower_percentile),
            'ci_upper': np.percentile(baseline_map50_bootstrap, upper_percentile),
            'std_error': np.std(baseline_map50_bootstrap)
        },
        'finetuned': {
            'point_estimate': finetuned_map50,
            'bootstrap_mean': np.mean(finetuned_map50_bootstrap),
            'bootstrap_distribution': finetuned_map50_bootstrap,
            'ci_lower': np.percentile(finetuned_map50_bootstrap, lower_percentile),
            'ci_upper': np.percentile(finetuned_map50_bootstrap, upper_percentile),
            'std_error': np.std(finetuned_map50_bootstrap)
        },
        'improvement': {
            'point_estimate': finetuned_map50 - baseline_map50,
            'bootstrap_mean': np.mean(delta_map50_bootstrap),
            'bootstrap_distribution': delta_map50_bootstrap,
            'ci_lower': np.percentile(delta_map50_bootstrap, lower_percentile),
            'ci_upper': np.percentile(delta_map50_bootstrap, upper_percentile),
            'std_error': np.std(delta_map50_bootstrap),
            'p_value': np.mean(delta_map50_bootstrap <= 0)
        },
        'config': {
            'n_bootstrap': n_bootstrap,
            'n_images': n_images,
            'confidence_level': confidence_level
        }
    }
    return results

## 6. Simplified Bootstrap Using YOLO Validation

A more practical approach: use YOLO's built-in validation but on bootstrap-sampled subsets.

## Choosing Your Bootstrap Method

**Two approaches are now available:**

### Option 1: Manual mAP50 Calculation (Fast, ~1-10 minutes for 1000 iterations)
- Function: `bootstrap_map50()`
- Loads predictions once, then resamples and recalculates mAP50 from stored predictions
- Pros: Much faster, uses manual IoU and AP calculation
- Cons: May have slight differences from YOLO's internal metric calculation
- **Recommended for: Initial exploration and testing**

### Option 2: YOLO Validation Bootstrap (Slow, ~hours for 1000 iterations)
- Function: `bootstrap_with_yolo_validation()`  
- Creates temporary dataset configs for each bootstrap sample and runs full YOLO validation
- Pros: Uses YOLO's exact metric calculation, guaranteed correctness
- Cons: Very computationally expensive (runs full validation N times)
- **Recommended for: Final results when accuracy is critical**

**For most cases, Option 1 provides reliable results much faster. Use Option 2 if you need to match YOLO's exact metrics.**

In [ ]:
# Choose your bootstrap method
# METHOD 1: Fast manual mAP50 calculation (recommended for testing)
# METHOD 2: Slow YOLO validation (more accurate but much slower)

USE_FAST_METHOD = True  # Set to False to use YOLO validation method
N_BOOTSTRAP = 1000  # Increase to 10000 for final results

if USE_FAST_METHOD:
    print("Using FAST manual mAP50 calculation method")
    print("=" * 80)
    
    # Check if we have ground truth loaded
    if len(ground_truth) == 0:
        print("WARNING: No ground truth loaded!")
        print("This likely means the labels path is incorrect.")
        print("Please check cell-10 and fix the load_ground_truth() function.")
        print("\nAttempting to fix by checking the data.yaml structure...")
        
        # Let's debug the path
        import yaml
        with open(data_yaml, 'r') as f:
            data_config = yaml.safe_load(f)
        print(f"\nData config: {data_config}")
        print("\nFalling back to YOLO validation method...")
        USE_FAST_METHOD = False
    else:
        bootstrap_results = bootstrap_map50(
            baseline_predictions=baseline_preds,
            finetuned_predictions=finetuned_preds,
            ground_truth=ground_truth,
            n_bootstrap=N_BOOTSTRAP,
            confidence_level=0.95,
            random_seed=42
        )

if not USE_FAST_METHOD:
    print("Using SLOW YOLO validation method (this will take a while...)")
    print("=" * 80)
    bootstrap_results = bootstrap_with_yolo_validation(
        baseline_model=baseline_model,
        finetuned_model=finetuned_model,
        data_yaml=data_yaml,
        test_images=test_images,
        n_bootstrap=N_BOOTSTRAP,
        confidence_level=0.95,
        random_seed=42
    )

print("\n✓ Bootstrap complete!")

Using FAST manual mAP50 calculation method
Running 1000 bootstrap iterations...
Note: This uses manual mAP50 calculation on resampled predictions.
For more accurate results, use bootstrap_with_yolo_validation() instead.



100%|██████████| 1000/1000 [25:10<00:00,  1.51s/it]


✓ Bootstrap complete!


In [ ]:
# Run bootstrap (choose appropriate number of iterations)
# Start with smaller number for testing, increase to 1000-10000 for final analysis
N_BOOTSTRAP = 100  # Increase to 1000-10000 for final results

bootstrap_results = bootstrap_with_yolo_validation(
    baseline_model=baseline_model,
    finetuned_model=finetuned_model,
    data_yaml=data_yaml,
    test_images=test_images,
    n_bootstrap=N_BOOTSTRAP,
    confidence_level=0.95,
    random_seed=42
)

NameError: name 'bootstrap_with_yolo_validation' is not defined

## 7. Results and Visualization

In [ ]:
def print_bootstrap_results(results):
    """
    Print formatted bootstrap confidence interval results.
    """
    print("="*80)
    print("BOOTSTRAP CONFIDENCE INTERVAL RESULTS")
    print("="*80)
    
    print(f"\nConfiguration:")
    print(f"  Number of bootstrap iterations: {results['config']['n_bootstrap']}")
    print(f"  Number of test images: {results['config']['n_images']}")
    print(f"  Confidence level: {results['config']['confidence_level']*100}%")
    
    print(f"\n" + "-"*80)
    print("BASELINE MODEL")
    print("-"*80)
    b = results['baseline']
    print(f"  Point Estimate:      {b['point_estimate']:.4f} ({b['point_estimate']*100:.2f}%)")
    print(f"  Bootstrap Mean:      {b['bootstrap_mean']:.4f} ({b['bootstrap_mean']*100:.2f}%)")
    print(f"  95% CI:              [{b['ci_lower']:.4f}, {b['ci_upper']:.4f}]")
    print(f"  95% CI (percent):    [{b['ci_lower']*100:.2f}%, {b['ci_upper']*100:.2f}%]")
    print(f"  Standard Error:      {b['std_error']:.4f}")
    
    print(f"\n" + "-"*80)
    print("FINE-TUNED MODEL")
    print("-"*80)
    f = results['finetuned']
    print(f"  Point Estimate:      {f['point_estimate']:.4f} ({f['point_estimate']*100:.2f}%)")
    print(f"  Bootstrap Mean:      {f['bootstrap_mean']:.4f} ({f['bootstrap_mean']*100:.2f}%)")
    print(f"  95% CI:              [{f['ci_lower']:.4f}, {f['ci_upper']:.4f}]")
    print(f"  95% CI (percent):    [{f['ci_lower']*100:.2f}%, {f['ci_upper']*100:.2f}%]")
    print(f"  Standard Error:      {f['std_error']:.4f}")
    
    print(f"\n" + "-"*80)
    print("IMPROVEMENT (Fine-tuned - Baseline)")
    print("-"*80)
    d = results['improvement']
    print(f"  Point Estimate:      {d['point_estimate']:.4f} ({d['point_estimate']*100:.2f}%)")
    print(f"  Bootstrap Mean:      {d['bootstrap_mean']:.4f} ({d['bootstrap_mean']*100:.2f}%)")
    print(f"  95% CI:              [{d['ci_lower']:.4f}, {d['ci_upper']:.4f}]")
    print(f"  95% CI (percent):    [{d['ci_lower']*100:.2f}%, {d['ci_upper']*100:.2f}%]")
    print(f"  Standard Error:      {d['std_error']:.4f}")
    print(f"  P-value:             {d['p_value']:.4f}")
    
    # Interpretation
    print(f"\n" + "="*80)
    print("INTERPRETATION")
    print("="*80)
    
    if d['ci_lower'] > 0:
        print(f"\n✓ The 95% confidence interval for improvement EXCLUDES zero.")
        print(f"  This provides strong evidence that fine-tuning improved performance.")
    elif d['ci_upper'] < 0:
        print(f"\n✗ The 95% confidence interval suggests fine-tuning may have decreased performance.")
    else:
        print(f"\n⚠ The 95% confidence interval INCLUDES zero.")
        print(f"  This suggests the observed improvement may not be statistically significant.")
        print(f"  More data or further fine-tuning may be needed.")
    
    print(f"\nStatistical note:")
    print(f"  The CI is constructed via paired bootstrap resampling over test images.")
    print(f"  This naturally accounts for:")
    print(f"    - The complex, non-linear nature of mAP50 as a statistic")
    print(f"    - Correlation between models (both evaluated on same images)")
    print(f"    - Sampling variability in the finite test set")
    
    print("\n" + "="*80)


# Print results
print_bootstrap_results(bootstrap_results)

BOOTSTRAP CONFIDENCE INTERVAL RESULTS

Configuration:
  Number of bootstrap iterations: 1000
  Number of test images: 481
  Confidence level: 95.0%

--------------------------------------------------------------------------------
BASELINE MODEL
--------------------------------------------------------------------------------
  Point Estimate:      0.0014 (0.14%)
  Bootstrap Mean:      0.0000 (0.00%)
  95% CI:              [0.0000, 0.0000]
  95% CI (percent):    [0.00%, 0.00%]
  Standard Error:      0.0000

--------------------------------------------------------------------------------
FINE-TUNED MODEL
--------------------------------------------------------------------------------
  Point Estimate:      0.8699 (86.99%)
  Bootstrap Mean:      0.0000 (0.00%)
  95% CI:              [0.0000, 0.0000]
  95% CI (percent):    [0.00%, 0.00%]
  Standard Error:      0.0000

--------------------------------------------------------------------------------
IMPROVEMENT (Fine-tuned - Baseline)
-------

In [ ]:
def plot_bootstrap_distributions(results, save_path=None):
    """
    Visualize bootstrap distributions and confidence intervals.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Bootstrap Confidence Intervals for mAP50', fontsize=16, fontweight='bold')
    
    # 1. Baseline distribution
    ax = axes[0, 0]
    b = results['baseline']
    ax.hist(b['bootstrap_distribution'], bins=50, alpha=0.7, color='blue', edgecolor='black')
    ax.axvline(b['point_estimate'], color='red', linestyle='--', linewidth=2, label='Point Estimate')
    ax.axvline(b['ci_lower'], color='green', linestyle=':', linewidth=2, label='95% CI')
    ax.axvline(b['ci_upper'], color='green', linestyle=':', linewidth=2)
    ax.set_xlabel('mAP50')
    ax.set_ylabel('Frequency')
    ax.set_title('Baseline Model Bootstrap Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Fine-tuned distribution
    ax = axes[0, 1]
    f = results['finetuned']
    ax.hist(f['bootstrap_distribution'], bins=50, alpha=0.7, color='orange', edgecolor='black')
    ax.axvline(f['point_estimate'], color='red', linestyle='--', linewidth=2, label='Point Estimate')
    ax.axvline(f['ci_lower'], color='green', linestyle=':', linewidth=2, label='95% CI')
    ax.axvline(f['ci_upper'], color='green', linestyle=':', linewidth=2)
    ax.set_xlabel('mAP50')
    ax.set_ylabel('Frequency')
    ax.set_title('Fine-tuned Model Bootstrap Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Improvement distribution
    ax = axes[1, 0]
    d = results['improvement']
    ax.hist(d['bootstrap_distribution'], bins=50, alpha=0.7, color='green', edgecolor='black')
    ax.axvline(d['point_estimate'], color='red', linestyle='--', linewidth=2, label='Point Estimate')
    ax.axvline(0, color='black', linestyle='-', linewidth=1, label='No Improvement')
    ax.axvline(d['ci_lower'], color='blue', linestyle=':', linewidth=2, label='95% CI')
    ax.axvline(d['ci_upper'], color='blue', linestyle=':', linewidth=2)
    ax.set_xlabel('Improvement (Fine-tuned - Baseline)')
    ax.set_ylabel('Frequency')
    ax.set_title('Improvement Bootstrap Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. Comparison with CIs
    ax = axes[1, 1]
    models = ['Baseline', 'Fine-tuned']
    estimates = [b['point_estimate'], f['point_estimate']]
    ci_lower = [b['ci_lower'], f['ci_lower']]
    ci_upper = [b['ci_upper'], f['ci_upper']]
    
    x_pos = np.arange(len(models))
    ax.scatter(x_pos, estimates, s=100, color=['blue', 'orange'], zorder=3, label='Point Estimate')
    ax.errorbar(x_pos, estimates, 
                yerr=[np.array(estimates) - np.array(ci_lower), 
                      np.array(ci_upper) - np.array(estimates)],
                fmt='none', capsize=10, capthick=2, elinewidth=2, 
                color='black', zorder=2, label='95% CI')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(models)
    ax.set_ylabel('mAP50')
    ax.set_title('Model Comparison with Confidence Intervals')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\nPlot saved to: {save_path}")
    
    plt.show()


# Create visualization
output_plot = Path('../data/output/bootstrap_confidence_intervals.png')
output_plot.parent.mkdir(parents=True, exist_ok=True)
plot_bootstrap_distributions(bootstrap_results, save_path=output_plot)

## 8. Save Results

In [ ]:
# Save results to JSON
output_json = Path('../data/output/bootstrap_results.json')

# Convert numpy arrays to lists for JSON serialization
results_serializable = {
    'baseline': {
        'point_estimate': float(results['baseline']['point_estimate']),
        'bootstrap_mean': float(results['baseline']['bootstrap_mean']),
        'ci_lower': float(results['baseline']['ci_lower']),
        'ci_upper': float(results['baseline']['ci_upper']),
        'std_error': float(results['baseline']['std_error'])
    },
    'finetuned': {
        'point_estimate': float(results['finetuned']['point_estimate']),
        'bootstrap_mean': float(results['finetuned']['bootstrap_mean']),
        'ci_lower': float(results['finetuned']['ci_lower']),
        'ci_upper': float(results['finetuned']['ci_upper']),
        'std_error': float(results['finetuned']['std_error'])
    },
    'improvement': {
        'point_estimate': float(results['improvement']['point_estimate']),
        'bootstrap_mean': float(results['improvement']['bootstrap_mean']),
        'ci_lower': float(results['improvement']['ci_lower']),
        'ci_upper': float(results['improvement']['ci_upper']),
        'std_error': float(results['improvement']['std_error']),
        'p_value': float(results['improvement']['p_value'])
    },
    'config': results['config']
}

with open(output_json, 'w') as f:
    json.dump(results_serializable, f, indent=2)

print(f"Results saved to: {output_json}")

# Also save full bootstrap distributions
output_npz = Path('../data/output/bootstrap_distributions.npz')
np.savez(
    output_npz,
    baseline=results['baseline']['bootstrap_distribution'],
    finetuned=results['finetuned']['bootstrap_distribution'],
    improvement=results['improvement']['bootstrap_distribution']
)

print(f"Bootstrap distributions saved to: {output_npz}")

## Summary

This notebook implements bootstrap confidence intervals for comparing object detection model performance using mAP50. The key advantages of this approach:

1. **Handles complex statistics**: mAP50 is not a simple mean; bootstrap naturally handles its complexity
2. **Paired comparison**: Uses the same resampled images for both models, preserving correlation
3. **No distributional assumptions**: Nonparametric approach that doesn't assume mAP50 follows any particular distribution
4. **Direct inference**: Provides CIs directly on the quantity of interest (improvement)

### Recommended Reporting

For your thesis/paper:

> "Model performance was evaluated using mAP50 on a test set of N images. To quantify uncertainty, we computed 95% confidence intervals via paired bootstrap resampling with 10,000 iterations. For each bootstrap sample, we resampled images with replacement (maintaining pairing across models) and recomputed mAP50.
>
> Results:
> - Baseline model: mAP50 = X.XX [95% CI: A.AA, B.BB]
> - Fine-tuned model: mAP50 = Y.YY [95% CI: C.CC, D.DD]
> - Improvement: ΔmAP50 = Z.ZZ [95% CI: E.EE, F.FF]
>
> The confidence interval for improvement excludes zero, providing strong evidence that fine-tuning significantly improved detection performance."